# Agrupamiento (Clustering) - Dataset FIRE UdeA

**SI3015 - Fundamentos de Aprendizaje Automático**

Aplicamos K-Means y DBSCAN sobre el dataset `dataset_para_modelado.csv` (variables financieras de empresas).

**Proceso:**
1. Carga e inspección del dataset
2. K-Means con K=2
3. Método del codo para hallar el mejor K
4. K-Means con el K óptimo
5. DBSCAN
6. UMAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

plt.rc('font', family='serif', size=12)
sns.set_theme(style='whitegrid', palette='Set2')

random_state = 42

## 1. Carga e inspección del dataset

In [ ]:
df = pd.read_csv(r'C:\Users\iidar\OneDrive\Escritorio\sexto-semestre\Aprendizaje\isabella-idarraga-botero\lecture8\dataset_para_modelado.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Separamos las features (excluimos 'label' — es la variable supervisada)
feature_cols = ['cfo', 'dias_efectivo', 'gp_ratio', 'endeudamiento',
                'liquidez', 'tendencia_ingresos', 'hhi_fuentes']

data = df[feature_cols].values
print(f'Features: {feature_cols}')
print(f'Shape datos: {data.shape}')

In [ ]:
# Distribución de las features
df[feature_cols].describe()

## 2. Pipeline de preprocesamiento

Escalamos con `StandardScaler` dado que las variables tienen escalas muy distintas (ej. `cfo` en miles de millones vs `gp_ratio` entre 0 y 1).

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, np.arange(data.shape[1])),
])

# También escalamos los datos una vez para reusar en DBSCAN y UMAP
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

## 3. K-Means con K = 2

In [ ]:
clu_kmeans_2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clustering', KMeans(n_clusters=2, random_state=random_state))
])

clu_kmeans_2.fit(data)
labels_k2 = clu_kmeans_2['clustering'].labels_
print(f'Inercia con K=2: {clu_kmeans_2["clustering"].inertia_:.2f}')
print(f'Distribución de clusters: {np.unique(labels_k2, return_counts=True)}')

In [ ]:
# Visualizamos con 2 pares de variables relevantes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['endeudamiento'], df['liquidez'], c=labels_k2, cmap='Set1', alpha=0.6, s=15)
axes[0].set_xlabel('Endeudamiento')
axes[0].set_ylabel('Liquidez')
axes[0].set_title('K-Means K=2: Endeudamiento vs Liquidez')

axes[1].scatter(df['gp_ratio'], df['tendencia_ingresos'], c=labels_k2, cmap='Set1', alpha=0.6, s=15)
axes[1].set_xlabel('GP Ratio')
axes[1].set_ylabel('Tendencia Ingresos')
axes[1].set_title('K-Means K=2: GP Ratio vs Tendencia Ingresos')

plt.tight_layout()
plt.show()

## 4. Método del codo — Hallar el K óptimo

In [ ]:
inert = []
k_range = list(range(1, 11))

for k in k_range:
    clu = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('clustering', KMeans(n_clusters=k, random_state=random_state))
    ])
    clu.fit(data)
    inert.append(clu['clustering'].inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inert, marker='o')
ax.set_xlabel('Número de clusters K')
ax.set_ylabel('Inercia')
ax.set_title('Método del Codo - Dataset FIRE UdeA')
ax.set_xticks(k_range)
plt.tight_layout()
plt.show()

## 5. K-Means con el K óptimo

Observa el gráfico anterior y ajusta `K_OPTIMO` al valor donde se encuentra el "codo".

In [ ]:
K_OPTIMO = 3  # <-- ajusta según el gráfico del codo

clu_kmeans_opt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clustering', KMeans(n_clusters=K_OPTIMO, random_state=random_state))
])

clu_kmeans_opt.fit(data)
labels_opt = clu_kmeans_opt['clustering'].labels_
print(f'Inercia con K={K_OPTIMO}: {clu_kmeans_opt["clustering"].inertia_:.2f}')
print(f'Distribución de clusters: {np.unique(labels_opt, return_counts=True)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['endeudamiento'], df['liquidez'], c=labels_opt, cmap='Set1', alpha=0.6, s=15)
axes[0].set_xlabel('Endeudamiento')
axes[0].set_ylabel('Liquidez')
axes[0].set_title(f'K-Means K={K_OPTIMO}: Endeudamiento vs Liquidez')

axes[1].scatter(df['gp_ratio'], df['tendencia_ingresos'], c=labels_opt, cmap='Set1', alpha=0.6, s=15)
axes[1].set_xlabel('GP Ratio')
axes[1].set_ylabel('Tendencia Ingresos')
axes[1].set_title(f'K-Means K={K_OPTIMO}: GP Ratio vs Tendencia Ingresos')

plt.tight_layout()
plt.show()

## 6. DBSCAN

DBSCAN no requiere especificar K y detecta outliers (etiqueta `-1`).

In [ ]:
clu_dbscan = DBSCAN(eps=0.5, min_samples=10)
clu_dbscan.fit(data_scaled)

labels_dbscan = clu_dbscan.labels_
clusters_unicos, conteos = np.unique(labels_dbscan, return_counts=True)
print('Clusters encontrados (label=-1 son outliers):')
for c, n in zip(clusters_unicos, conteos):
    nombre = 'Outliers' if c == -1 else f'Cluster {c}'
    print(f'  {nombre}: {n} puntos')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['endeudamiento'], df['liquidez'], c=labels_dbscan, cmap='Set1', alpha=0.6, s=15)
axes[0].set_xlabel('Endeudamiento')
axes[0].set_ylabel('Liquidez')
axes[0].set_title('DBSCAN: Endeudamiento vs Liquidez')

axes[1].scatter(df['gp_ratio'], df['tendencia_ingresos'], c=labels_dbscan, cmap='Set1', alpha=0.6, s=15)
axes[1].set_xlabel('GP Ratio')
axes[1].set_ylabel('Tendencia Ingresos')
axes[1].set_title('DBSCAN: GP Ratio vs Tendencia Ingresos')

plt.tight_layout()
plt.show()

## 7. UMAP

UMAP reduce las 7 dimensiones del dataset a 2D para visualizar la estructura real de los datos.
Coloreamos el embedding con el **label real**, **K-Means** y **DBSCAN** para comparar.

> Si no tienes instalado umap: `pip install umap-learn`

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=random_state)
embedding = reducer.fit_transform(data_scaled)

print(f'Shape embedding UMAP: {embedding.shape}')  # (n_empresas, 2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    (df['label'].values, 'Label real'),
    (labels_k2,          'K-Means K=2'),
    (labels_opt,         f'K-Means K={K_OPTIMO}'),
]

for ax, (labels, titulo) in zip(axes, configs):
    sc = ax.scatter(embedding[:, 0], embedding[:, 1],
                    c=labels, cmap='Set1', alpha=0.6, s=10)
    ax.set_title(titulo)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    plt.colorbar(sc, ax=ax)

plt.suptitle('UMAP - Dataset FIRE UdeA', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# UMAP coloreado con DBSCAN
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(embedding[:, 0], embedding[:, 1],
                c=labels_dbscan, cmap='Set1', alpha=0.6, s=10)
ax.set_title('UMAP - DBSCAN (gris = outliers)')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.colorbar(sc, ax=ax)
plt.tight_layout()
plt.show()

## 8. Comparación con el label real (variables originales)

Como ejercicio adicional, comparamos los clusters encontrados con la variable `label` original del dataset (no se usó en el clustering).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (labels, titulo) in zip(axes, [
    (df['label'].values, 'Label real'),
    (labels_k2,          f'K-Means K=2'),
    (labels_opt,         f'K-Means K={K_OPTIMO}'),
]):
    ax.scatter(df['endeudamiento'], df['liquidez'], c=labels, cmap='Set1', alpha=0.6, s=15)
    ax.set_xlabel('Endeudamiento')
    ax.set_ylabel('Liquidez')
    ax.set_title(titulo)

plt.suptitle('Comparación: label real vs clusters encontrados', fontsize=13)
plt.tight_layout()
plt.show()